# SSS Marine Debris Detection — Two-Stage Training

## Strategy

**Stage 1: COMPETITION** (Quick, ~30 epochs each)
- Train all 3 models: YOLOv8n, SS-YOLO, YOLOv8-ESI
- Evaluate on validation set
- Select winner by F1 score

**Stage 2: WINNER TRAINING** (Thorough, 150+ epochs)
- Train winner with optimized hyperparameters
- Extended patience, learning rate scheduling
- Confidence threshold sweep
- Final evaluation on test set

## Models
| Model | Params | Strategy | Key Feature |
|-------|--------|----------|-------------|
| YOLOv8n | 3.01M | Fine-tune | Baseline standard |
| SS-YOLO | 1.66M | From scratch | GhostConv + PConv (45% smaller) |
| YOLOv8-ESI | 3.18M | Fine-tune | SE attention for textures |

In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 1: Setup
# ═══════════════════════════════════════════════════════════
!pip install ultralytics pandas matplotlib -q
!git clone https://github.com/Dinoman67/sonarvision.git
%cd sonarvision

import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 2: Upload and balance dataset
# ═══════════════════════════════════════════════════════════
from google.colab import files
uploaded = files.upload()  # Upload e5.zip
!unzip -q e5.zip -d /content/

# Balance dataset (oversample debris 3x)
# If balance script exists, use it. Otherwise, skip balancing.
import os
if os.path.exists('scripts/balance_e5_dataset.py'):
    !python scripts/balance_e5_dataset.py \
        --e5 /content/e5 \
        --output /content/e5_balanced \
        --ratio 3
    DATA = '/content/e5_balanced/data.yaml'
    print(f'\nDataset balanced: {DATA}')
else:
    print('⚠ balance_e5_dataset.py not found — using unbalanced E5')
    print('  To fix: git pull to get latest scripts, or run manually:')
    print('  !python scripts/balance_e5_dataset.py --e5 /content/e5 --output /content/e5_balanced')
    DATA = '/content/e5/data.yaml'
    print(f'\nDataset ready: {DATA}')


In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 3: Load custom modules
# ═══════════════════════════════════════════════════════════
import sys
from pathlib import Path

from ultralytics import YOLO
from models.sss_custom_modules import (
    PConv, FasterBlock, FastC2f, GhostConv,
    SEBlock, CBAM, DepthwiseSeparableConv
)
from models.build_sss_models import build_ss_yolo, build_yolov8_esi_full, C2fWithSE
from ultralytics.models.yolo.detect.train import DetectionTrainer
print('✓ Custom modules loaded')

## ═══════════════════════════════════════════════════════════
## STAGE 1: MODEL COMPETITION
## ═══════════════════════════════════════════════════════════

Quick training (30 epochs) to find the best model architecture.

In [ ]:
# ═══════════════════════════════════════════════════════════
# Helper: Confidence sweep
# ═══════════════════════════════════════════════════════════
def evaluate_model(model, data_yaml, imgsz=512, conf=0.25, split='val'):
    """Evaluate model and return metrics."""
    r = model.val(data=data_yaml, imgsz=imgsz, conf=conf, split=split, verbose=False)
    p, rv = r.box.mp, r.box.mr
    f1 = 2*p*rv / max(p+rv, 1e-8)
    return {'mAP50': r.box.map50, 'P': p, 'R': rv, 'F1': f1}

def conf_sweep(model, data_yaml, imgsz=512):
    """Find optimal confidence threshold."""
    best_f1, best_conf = 0, 0.25
    for c in [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]:
        r = model.val(data=data_yaml, imgsz=imgsz, conf=c, verbose=False)
        p, rv = r.box.mp, r.box.mr
        f1 = 2*p*rv / max(p+rv, 1e-8)
        if f1 > best_f1:
            best_f1, best_conf = f1, c
    return best_conf

In [ ]:
# ═══════════════════════════════════════════════════════════
# Stage 1: Train YOLOv8n
# ═══════════════════════════════════════════════════════════
print('='*60)
print('STAGE 1: Training YOLOv8n (30 epochs)')
print('='*60)

model_v8n = YOLO('yolov8n.pt')
model_v8n.train(
    data=DATA, epochs=30, imgsz=512, batch=16, patience=15,
    lr0=0.01, lrf=0.01, warmup_epochs=2,
    mosaic=0.0, mixup=0.0,
    fliplr=0.0, flipud=0.0, degrees=0.0,
    translate=0.05, scale=0.2,
    name='yolov8n_s1', project='/content/runs', exist_ok=True, plots=True,
)

metrics_v8n = evaluate_model(model_v8n, DATA)
print(f'\n✓ YOLOv8n Stage 1: mAP50={metrics_v8n["mAP50"]:.4f}, '
      f'P={metrics_v8n["P"]:.4f}, R={metrics_v8n["R"]:.4f}, F1={metrics_v8n["F1"]:.4f}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# Stage 1: Train SS-YOLO (pretrained backbone!)
# ═══════════════════════════════════════════════════════════
print('='*60)
print('STAGE 1: Training SS-YOLO (30 epochs, pretrained backbone)')
print('='*60)

# SS-YOLO with pretrained backbone (not from scratch!)
# This uses YOLOv8n pretrained weights in backbone, only neck uses GhostConv+FastC2f
ss_model = build_ss_yolo(pretrained='yolov8n.pt', mode='backbone_only')

_orig = DetectionTrainer.get_model
def _patched_ss(self, cfg=None, weights=None, verbose=True):
    from ultralytics.nn.tasks import DetectionModel
    from ultralytics.utils import RANK
    model = self.set_model_names_for_load(
        DetectionModel(cfg, nc=self.data['nc'], ch=self.data['channels'],
                       verbose=verbose and RANK == -1)
    )
    model.model = ss_model.model
    model.nc = 1
    model.names = {0: 'marine_debris'}
    return model
DetectionTrainer.get_model = _patched_ss

try:
    yolo_ss = YOLO('yolov8n.pt')
    yolo_ss.train(
        data=DATA, epochs=30, imgsz=512, batch=16, patience=15,
        lr0=0.005, lrf=0.01, warmup_epochs=3,  # Lower LR for pretrained
        mosaic=0.0, mixup=0.0,
        fliplr=0.0, flipud=0.0, degrees=0.0,
        translate=0.05, scale=0.2,
        name='ss_yolo_s1', project='/content/runs', exist_ok=True, plots=True,
    )
    metrics_ss = evaluate_model(yolo_ss, DATA)
    print(f'\n✓ SS-YOLO Stage 1: mAP50={metrics_ss["mAP50"]:.4f}, '
          f'P={metrics_ss["P"]:.4f}, R={metrics_ss["R"]:.4f}, F1={metrics_ss["F1"]:.4f}')
except Exception as e:
    print(f'✗ SS-YOLO failed: {e}')
    import traceback; traceback.print_exc()
    metrics_ss = {'mAP50': 0, 'P': 0, 'R': 0, 'F1': 0}
    yolo_ss = None
finally:
    DetectionTrainer.get_model = _orig


In [ ]:
# ═══════════════════════════════════════════════════════════
# Stage 1: Train YOLOv8-ESI
# ═══════════════════════════════════════════════════════════
print('='*60)
print('STAGE 1: Training YOLOv8-ESI (30 epochs)')
print('='*60)

esi_model = build_yolov8_esi_full()

_orig2 = DetectionTrainer.get_model
def _patched_esi(self, cfg=None, weights=None, verbose=True):
    from ultralytics.nn.tasks import DetectionModel
    from ultralytics.utils import RANK
    model = self.set_model_names_for_load(
        DetectionModel(cfg, nc=self.data['nc'], ch=self.data['channels'],
                       verbose=verbose and RANK == -1)
    )
    model.model = esi_model.model
    model.nc = 1
    model.names = {0: 'marine_debris'}
    try:
        model.load(weights)
    except Exception:
        pass
    return model
DetectionTrainer.get_model = _patched_esi

try:
    yolo_esi = YOLO('yolov8n.pt')
    yolo_esi.train(
        data=DATA, epochs=30, imgsz=512, batch=16, patience=15,
        lr0=0.01, lrf=0.01, warmup_epochs=2,
        mosaic=0.0, mixup=0.0,
        fliplr=0.0, flipud=0.0, degrees=0.0,
        translate=0.05, scale=0.2,
        name='yolov8_esi_s1', project='/content/runs', exist_ok=True, plots=True,
    )
    metrics_esi = evaluate_model(yolo_esi, DATA)
    print(f'\n✓ YOLOv8-ESI Stage 1: mAP50={metrics_esi["mAP50"]:.4f}, '
          f'P={metrics_esi["P"]:.4f}, R={metrics_esi["R"]:.4f}, F1={metrics_esi["F1"]:.4f}')
except Exception as e:
    print(f'✗ YOLOv8-ESI failed: {e}')
    import traceback; traceback.print_exc()
    metrics_esi = {'mAP50': 0, 'P': 0, 'R': 0, 'F1': 0}
    yolo_esi = None
finally:
    DetectionTrainer.get_model = _orig2

In [ ]:
# ═══════════════════════════════════════════════════════════
# Stage 1: Results & Winner Selection
# ═══════════════════════════════════════════════════════════
import pandas as pd

all_models = {
    'YOLOv8n': {'model': model_v8n, 'metrics': metrics_v8n},
    'SS-YOLO': {'model': yolo_ss, 'metrics': metrics_ss},
    'YOLOv8-ESI': {'model': yolo_esi, 'metrics': metrics_esi},
}

print('\n' + '='*70)
print('STAGE 1: COMPETITION RESULTS')
print('='*70)

rows = []
for name, info in all_models.items():
    m = info['metrics']
    n = sum(p.numel() for p in info['model'].model.parameters()) if info['model'] else 0
    rows.append({
        'Model': name, 'Params': f'{n/1e6:.2f}M',
        'mAP50': m['mAP50'], 'P': m['P'], 'R': m['R'], 'F1': m['F1'],
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# Select winner
winner_name = df.loc[df['F1'].idxmax(), 'Model']
winner_info = all_models[winner_name]

print(f'\n{"="*70}')
print(f'🏆 WINNER: {winner_name} (F1={winner_info["metrics"]["F1"]:.4f})')
print(f'{"="*70}')

## ═══════════════════════════════════════════════════════════
## STAGE 2: WINNER TRAINING
## ═══════════════════════════════════════════════════════════

Training the winner for 150+ epochs with optimized hyperparameters.

In [ ]:
# ═══════════════════════════════════════════════════════════
# Stage 2: Train Winner
# ═══════════════════════════════════════════════════════════
print('='*60)
print(f'STAGE 2: Training {winner_name} (150+ epochs)')
print('='*60)

# Configure based on model type
if winner_name == 'SS-YOLO':
    # SS-YOLO with pretrained backbone — can train faster
    stage2_epochs = 100
    stage2_lr0 = 0.005
    stage2_warmup = 3
else:
    # Fine-tuning models
    stage2_epochs = 150
    stage2_lr0 = 0.005
    stage2_warmup = 3

print(f'  Epochs: {stage2_epochs}')
print(f'  LR: {stage2_lr0}')
print(f'  Warmup: {stage2_warmup}')

# Rebuild and train winner
if winner_name == 'SS-YOLO':
    winner_model = build_ss_yolo(pretrained='yolov8n.pt', mode='backbone_only')
    _orig3 = DetectionTrainer.get_model
    def _patched_winner(self, cfg=None, weights=None, verbose=True):
        from ultralytics.nn.tasks import DetectionModel
        from ultralytics.utils import RANK
        model = self.set_model_names_for_load(
            DetectionModel(cfg, nc=self.data['nc'], ch=self.data['channels'],
                           verbose=verbose and RANK == -1)
        )
        model.model = winner_model.model
        model.nc = 1
        model.names = {0: 'marine_debris'}
        return model
    DetectionTrainer.get_model = _patched_winner
else:
    winner_model = YOLO('yolov8n.pt')
    _orig3 = None

try:
    winner_model.train(
        data=DATA, epochs=stage2_epochs, imgsz=512, batch=16, patience=40,
        lr0=stage2_lr0, lrf=0.01, warmup_epochs=stage2_warmup,
        mosaic=0.0, mixup=0.0,
        fliplr=0.0, flipud=0.0, degrees=0.0,
        translate=0.05, scale=0.2,
        name=f'{winner_name.lower().replace("-","")}_s2_final',
        project='/content/runs', exist_ok=True, plots=True,
    )
except Exception as e:
    print(f'✗ Stage 2 training failed: {e}')
    import traceback; traceback.print_exc()
finally:
    if _orig3:
        DetectionTrainer.get_model = _orig3


In [ ]:
# ═══════════════════════════════════════════════════════════
# Stage 2: Final Evaluation + Confidence Sweep
# ═══════════════════════════════════════════════════════════
print('='*60)
print('FINAL EVALUATION')
print('='*60)

# Confidence sweep
print('\nConfidence threshold sweep (val set):')
print(f'{"Conf":>6} {"P":>8} {"R":>8} {"mAP50":>8} {"F1":>8}')
print('-' * 42)

best_f1, best_conf = 0, 0.25
for c in [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]:
    r = winner_model.val(data=DATA, imgsz=512, conf=c, verbose=False)
    p, rv = r.box.mp, r.box.mr
    f1 = 2*p*rv / max(p+rv, 1e-8)
    marker = ' ←' if f1 > best_f1 else ''
    print(f'{c:>6.2f} {p:>8.4f} {rv:>8.4f} {r.box.map50:>8.4f} {f1:>8.4f}{marker}')
    if f1 > best_f1:
        best_f1, best_conf = f1, c

print(f'\nOptimal confidence: {best_conf}')

# Final test evaluation
print('\n--- TEST SET (unseen) ---')
test_r = winner_model.val(data=DATA, imgsz=512, conf=best_conf, split='test')
p, r = test_r.box.mp, test_r.box.mr
f1 = 2*p*r / max(p+r, 1e-8)

print(f'\n{"="*60}')
print(f'FINAL RESULTS: {winner_name}')
print(f'{"="*60}')
print(f'  Test mAP50:  {test_r.box.map50:.4f}')
print(f'  Test P:      {p:.4f}')
print(f'  Test R:      {r:.4f}')
print(f'  Test F1:     {f1:.4f}')
print(f'  Best conf:   {best_conf}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# Export best model
# ═══════════════════════════════════════════════════════════
print('\n' + '='*60)
print('EXPORT')
print('='*60)

winner_model.export(format='onnx', imgsz=512)
print(f'\n✓ Model exported to ONNX')

# Download
from google.colab import files
model_dir = f'/content/runs/{winner_name.lower().replace("-","")}_s2_final'
files.download(f'{model_dir}/weights/best.pt')
files.download(f'{model_dir}/weights/best.onnx')